# Chương 6: Thực hành - Tối ưu hóa và Giải tích cho Học máy (Optimization & Calculus for ML)

**Mục tiêu bài thực hành:**
1. **Tác động của Tốc độ học (Learning Rate):** Trực quan hóa hiện tượng hội tụ chậm, hội tụ tối ưu và phân kỳ nhảy cóc (overshooting) trên hàm số 1 chiều.
2. **Thực hiện từ đầu (from scratch) 4 thuật toán tối ưu hóa:** Vanilla Gradient Descent, Momentum, RMSProp và **Adam**.
3. **Thử thách trên Địa hình hiểm trở (Hẻm núi Ill-conditioned Valley):** Đối chiếu quỹ đạo di chuyển của các bộ tối ưu hóa trên cùng mặt phẳng đường mức 2D.
4. **So sánh 3 biến thể:** Batch GD vs. Stochastic GD (SGD) vs. Mini-batch GD trên bài toán Hồi quy tuyến tính.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Thiết lập hạt giống ngẫu nhiên
np.random.seed(42)
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')

## 1. Khảo sát Tác động của Tốc độ học (Learning Rate $\eta$) trên Hàm 1 chiều

Hàm mục tiêu: $f(x) = x^2$, đạo hàm $f'(x) = 2x$. Cực tiểu tại $x^* = 0$.
Quy tắc cập nhật: $x_{t+1} = x_t - \eta (2x_t) = (1 - 2\eta) x_t$.

In [ ]:
def gradient_descent_1d(eta, x_start=4.0, n_steps=12):
    history = [x_start]
    x = x_start
    for _ in range(n_steps):
        grad = 2.0 * x
        x = x - eta * grad
        history.append(x)
    return np.array(history)

etas = [(0.05, "Quá nhỏ (Chậm)", "#2980b9"), 
        (0.40, "Tối ưu (Hội tụ êm)", "#27ae60"), 
        (1.03, "Quá lớn (Phân kỳ)", "#c0392b")]

x_curve = np.linspace(-5, 5, 200)
y_curve = x_curve**2

fig, axes = plt.subplots(1, 3, figsize=(18, 4.5))
for idx, (lr, label, color) in enumerate(etas):
    ax = axes[idx]
    path = gradient_descent_1d(lr)
    ax.plot(x_curve, y_curve, 'k-', alpha=0.6, linewidth=1.5)
    ax.plot(path, path**2, 'o--', color=color, linewidth=1.8, markersize=6, label=f'eta = {lr}')
    ax.set_title(f"{label} (eta = {lr})", fontsize=11, fontweight='bold')
    ax.set_xlabel("x")
    ax.set_ylabel("f(x)")
    ax.set_ylim(-2, 25)
    ax.legend()

plt.suptitle("Tác động của Tốc độ học (Learning Rate) lên Quá trình Hội tụ 1D", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## 2. Thử thách trên Địa hình Hẻm núi 2D (Ill-Conditioned Valley)

Hàm mục tiêu mô phỏng hẻm núi hẹp: $f(x, y) = 0.1 x^2 + 2.0 y^2$.  
Gradient: $\nabla f(x, y) = [0.2x, 4.0y]^T$.  
Độ dốc theo chiều $y$ gấp 20 lần theo chiều $x$, khiến Vanilla Gradient Descent bị dao động ziczac rất mạnh giữa hai sườn dốc mà tiến rất chậm dọc theo lòng máng.

In [ ]:
def loss_fn(x, y):
    return 0.1 * x**2 + 2.0 * y**2

def grad_fn(x, y):
    return np.array([0.2 * x, 4.0 * y])

# 1. Vanilla Gradient Descent
def optimize_gd(lr=0.45, n_steps=60, start=[-4.5, 2.0]):
    pos = np.array(start, dtype=float)
    history = [pos.copy()]
    for _ in range(n_steps):
        pos -= lr * grad_fn(pos[0], pos[1])
        history.append(pos.copy())
    return np.array(history)

# 2. Gradient Descent with Momentum
def optimize_momentum(lr=0.1, gamma=0.85, n_steps=60, start=[-4.5, 2.0]):
    pos = np.array(start, dtype=float)
    v = np.zeros(2)
    history = [pos.copy()]
    for _ in range(n_steps):
        v = gamma * v + lr * grad_fn(pos[0], pos[1])
        pos -= v
        history.append(pos.copy())
    return np.array(history)

# 3. RMSProp
def optimize_rmsprop(lr=0.3, beta=0.9, eps=1e-8, n_steps=60, start=[-4.5, 2.0]):
    pos = np.array(start, dtype=float)
    v = np.zeros(2)
    history = [pos.copy()]
    for _ in range(n_steps):
        g = grad_fn(pos[0], pos[1])
        v = beta * v + (1 - beta) * (g**2)
        pos -= (lr / (np.sqrt(v) + eps)) * g
        history.append(pos.copy())
    return np.array(history)

# 4. Adam (Adaptive Moment Estimation)
def optimize_adam(lr=0.4, beta1=0.9, beta2=0.999, eps=1e-8, n_steps=60, start=[-4.5, 2.0]):
    pos = np.array(start, dtype=float)
    m = np.zeros(2)
    v = np.zeros(2)
    history = [pos.copy()]
    for t in range(1, n_steps + 1):
        g = grad_fn(pos[0], pos[1])
        m = beta1 * m + (1 - beta1) * g
        v = beta2 * v + (1 - beta2) * (g**2)
        m_hat = m / (1 - beta1**t)
        v_hat = v / (1 - beta2**t)
        pos -= (lr / (np.sqrt(v_hat) + eps)) * m_hat
        history.append(pos.copy())
    return np.array(history)

path_gd = optimize_gd()
path_mom = optimize_momentum()
path_rms = optimize_rmsprop()
path_adam = optimize_adam()

## 3. Trực quan hóa So sánh Quỹ đạo Tối ưu hóa (Optimization Trajectories 2D)

In [ ]:
# Tạo lưới vẽ đường mức địa hình
x_grid = np.linspace(-5, 5, 200)
y_grid = np.linspace(-3, 3, 200)
X, Y = np.meshgrid(x_grid, y_grid)
Z = loss_fn(X, Y)

plt.figure(figsize=(12, 6.5))
plt.contour(X, Y, Z, levels=np.logspace(-1, 2, 20), cmap='gray', alpha=0.6)

plt.plot(path_gd[:, 0], path_gd[:, 1], 'o-', color='#e74c3c', label='Vanilla GD (Bị dao động ziczac ngang)', markersize=3, linewidth=1.5)
plt.plot(path_mom[:, 0], path_mom[:, 1], 's-', color='#3498db', label='Momentum (Triệt tiêu rung lắc, lao nhanh)', markersize=3, linewidth=1.8)
plt.plot(path_rms[:, 0], path_rms[:, 1], '^-', color='#f39c12', label='RMSProp (Chuẩn hóa bước nhảy theo trục)', markersize=3, linewidth=1.8)
plt.plot(path_adam[:, 0], path_adam[:, 1], 'd-', color='#2ecc71', label='Adam (Hội tụ êm ái và trực diện nhất)', markersize=4, linewidth=2.2)

plt.scatter(0, 0, color='gold', s=180, marker='*', edgecolors='k', zorder=10, label='Cực tiểu toàn cục (0, 0)')
plt.title("So sánh Quỹ đạo Hội tụ: GD vs Momentum vs RMSProp vs Adam", fontsize=13, fontweight='bold')
plt.xlabel("Trục X (Độ dốc thoải)", fontsize=11)
plt.ylabel("Trục Y (Độ dốc gắt)", fontsize=11)
plt.xlim(-5, 5)
plt.ylim(-3, 3)
plt.legend(loc='lower left', fontsize=10)
plt.tight_layout()
plt.show()

## 4. Đối chiếu Thực nghiệm: Batch GD vs. Stochastic GD vs. Mini-batch GD

In [ ]:
# Sinh dữ liệu bài toán hồi quy: y = 2*x + 1 + noise
N = 1000
X_data = np.random.randn(N, 1)
y_data = 2.5 * X_data.squeeze() + 1.2 + np.random.randn(N) * 0.5

def train_variants(batch_size, lr=0.02, n_epochs=15):
    w = 0.0
    b = 0.0
    loss_history = []
    
    n_batches = N // batch_size
    for epoch in range(n_epochs):
        # Xáo trộn dữ liệu mỗi epoch
        indices = np.random.permutation(N)
        X_shuffled = X_data[indices]
        y_shuffled = y_data[indices]
        
        for b_idx in range(n_batches):
            start_i = b_idx * batch_size
            end_i = start_i + batch_size
            X_b = X_shuffled[start_i:end_i].squeeze()
            y_b = y_shuffled[start_i:end_i]
            
            # Tính dự đoán và gradient
            y_pred = w * X_b + b
            error = y_pred - y_b
            
            dw = np.mean(error * X_b)
            db = np.mean(error)
            
            w -= lr * dw
            b -= lr * db
            
            # Ghi nhận hàm mất mát toàn cục
            total_loss = np.mean((w * X_data.squeeze() + b - y_data)**2)
            loss_history.append(total_loss)
            
    return loss_history, w, b

loss_bgd, w_b, b_b = train_variants(batch_size=N, lr=0.08)       # Batch GD
loss_sgd, w_s, b_s = train_variants(batch_size=1, lr=0.01)       # Stochastic GD
loss_mini, w_m, b_m = train_variants(batch_size=32, lr=0.03)     # Mini-batch GD (32)

plt.figure(figsize=(12, 5))
plt.plot(loss_bgd[:150], label=f'Batch GD (B=1000, Rất mượt)', color='#2c3e50', linewidth=2)
plt.plot(loss_mini[:150], label=f'Mini-batch GD (B=32, Tối ưu)', color='#27ae60', linewidth=1.8)
plt.plot(loss_sgd[:150], label=f'SGD (B=1, Nhiễu dao động cao)', color='#e74c3c', alpha=0.6, linewidth=1)

plt.title("So sánh Tốc độ và Tính Ổn định của Loss: BGD vs SGD vs Mini-batch", fontsize=13, fontweight='bold')
plt.xlabel("Số bước cập nhật (Steps)", fontsize=11)
plt.ylabel("Mean Squared Error Loss", fontsize=11)
plt.yscale('log')
plt.legend(fontsize=11)
plt.tight_layout()
plt.show()

## 5. Tổng kết bài học thực hành

1. **Tốc độ học (Learning Rate):** Siêu tham số nhạy cảm nhất. $\eta$ quá lớn sẽ khiến mô hình nổ tung phân kỳ, $\eta$ quá nhỏ gây lãng phí năng lực tính toán.
2. **Momentum:** Tích lũy xung lượng theo hướng dốc chính, triệt tiêu rung lắc hai bên thành núi hẹp.
3. **RMSProp & Adam:** Tự động điều chỉnh bước nhảy độc lập cho từng tham số, đóng vai trò sống còn trong việc huấn luyện các mô hình học sâu hiện đại (CNNs, Transformers).
4. **Mini-batch GD:** Sự kết hợp hoàn hảo giữa tính ổn định thống kê của Batch GD và tính kháng bẫy điểm yên ngựa của SGD, đồng thời tận dụng triệt để năng lực tính toán song song của GPU.